In [22]:
import stim
import sinter
from typing import List
import matplotlib.pyplot as plt

In [23]:
def noisify(circuit, noise = 0.001, drop_ticks = False):
    noisy_circuit = stim.Circuit()

    for instruction in circuit.flattened():
        if instruction.name in ["CX", "CZ"]:
            noisy_circuit.append(instruction)
            noisy_circuit.append("DEPOLARIZE2", instruction.targets_copy(), noise)
        elif instruction.name in ["M", "MX"]:
            noisy_circuit.append(instruction.name, instruction.targets_copy(), noise)
        elif instruction.name in ["R", "RX"]:
            noisy_circuit.append(instruction)
            noisy_circuit.append("DEPOLARIZE1", instruction.targets_copy(), noise)
        elif instruction.name in ["QUBIT_COORDS", "DETECTOR", "OBSERVABLE_INCLUDE"]:
            noisy_circuit.append(instruction)
        elif instruction.name == "TICK":
            if not drop_ticks:
                noisy_circuit.append(instruction)
        else:
            raise NotImplementedError(f"Incomplete noisification : {instruction.name}")

    noisy_circuit.compile_detector_sampler()
    noisy_circuit.compile_sampler()

    return noisy_circuit

In [30]:
# circuit = stim.Circuit().from_file("../assets/tqec-extended-stabilizers.stim")
# from tqecd import annotate_detectors_automatically
# circuit = annotate_detectors_automatically(circuit)
# circuit.to_file("../assets/tqec-extended-stabilizers-auto-detectors.stim")

In [33]:
circuit = noisify(stim.Circuit().from_file("../assets/tqec-extended-stabilizers-auto-detectors.stim"))
errors = circuit.shortest_graphlike_error(canonicalize_circuit_errors=True)

print(f"Length of shortest graph-like error : {len(errors)}")
for error in errors:
    print(error) #str(error).split('\n'))[3][8:])

Length of shortest graph-like error : 1
ExplainedError {
    dem_error_terms: L0
    CircuitErrorLocation {
        flipped_measurement.measurement_record_index: 660
        flipped_measurement.measured_observable: Z0[coords 7,1]
        Circuit location stack trace:
            (after 42 TICKs)
            at instruction #675 (M) in the circuit
            at target #1 of the instruction
            resolving to M(0.001) 0[coords 7,1]
    }
}


In [ ]:
tasks = [
    sinter.Task(
        circuit=noisify(
            stim.Circuit().from_file("../assets/tqec-extended-stabilizers-detectors.stim"), noise=noise
        ),
        json_metadata={'d': d, 'p': noise},
    )
    for d in [5]
    for noise in [0.0001, 0.001, 0.003125, 0.00625, 0.0125, 0.025, 0.05, 0.08, 0.1]
]

collected_stats: List[sinter.TaskStats] = sinter.collect(
    num_workers=4,
    tasks=tasks,
    decoders=['pymatching'],
    max_shots=1_000_000,
    max_errors=500,
)

In [ ]:
fig, ax = plt.subplots(1, 1)
sinter.plot_error_rate(
    ax=ax,
    stats=collected_stats,
    x_func=lambda stats: stats.json_metadata['p'],
    group_func=lambda stats: stats.json_metadata['d'],
)
ax.set_ylim(1e-9, 1e-0)
ax.set_xlim(9e-4, 1.2e-1)
ax.loglog()
ax.set_title("Proposed Spatial Junction")
ax.set_xlabel("Phyical Error Rate")
ax.set_ylabel("Logical Error Rate per Shot")
ax.grid(which='major')
ax.grid(which='minor')
ax.legend()
fig.set_dpi(120)